# Zeros de Funções

### Métodos Numéricos — Engenharia Naval e Oceânica, Poli-USP

Este notebook é um material de apoio para estudo do tópico **Zeros de Funções** (ou *raízes de equações*). O objetivo é reunir, em um só lugar, teoria, implementação em Python "do zero" e visualizações que ajudam a entender **por que** cada método funciona (e quando falha).

**Métodos cobertos:**
1. Fundamentação teórica (Teorema do Valor Médio / Teorema de Bolzano)
2. Método da Dicotomia (Bisseção)
3. Método da Falsa Posição (Regula Falsi)
4. Método de Newton-Raphson
5. Método das Aproximações Sucessivas (Ponto Fixo)
6. Comparação final entre os métodos

> **Como usar este notebook:** execute as células em ordem, de cima para baixo. Os gráficos foram pensados para reforçar a intuição geométrica de cada método — pare e observe cada figura antes de seguir para a próxima seção.

## 1. Introdução e motivação

Um **zero** (ou **raiz**) de uma função $f$ é um valor $x^*$ tal que

$$f(x^*) = 0.$$

Encontrar raízes é um dos problemas mais recorrentes em engenharia: calcular o calado de equilíbrio de um navio, a velocidade de um escoamento onde o número de Reynolds crítico é atingido, o ponto de operação de um sistema não-linear, etc. Em todos esses casos, queremos resolver uma equação do tipo $f(x) = 0$.

**Por que não resolver analiticamente?** Para funções polinomiais de grau até 4 existem fórmulas fechadas (Bhaskara, Cardano, Ferrari). Mas a grande maioria das equações de interesse prático envolve funções **transcendentais** — combinações de exponenciais, logaritmos, funções trigonométricas — para as quais **não existe fórmula algébrica geral**. Por exemplo:

$$e^{-x} - \sin(x) = 0, \qquad x - \cos(x) = 0, \qquad x^5 - 3x + 1 = 0$$

Nenhuma dessas equações pode ser resolvida isolando $x$ algebricamente. Mesmo assim, sabemos que soluções existem e precisamos encontrá-las com a precisão que a aplicação de engenharia exigir.

**A ideia dos métodos numéricos:** em vez de buscar uma solução exata e fechada, construímos **sequências de aproximações** $x_0, x_1, x_2, \dots$ que convergem para a raiz $x^*$, parando quando a aproximação for "boa o suficiente" (dentro de uma tolerância pré-estabelecida). Esse é o tema central deste notebook: vamos estudar diferentes estratégias para construir essa sequência, cada uma com vantagens, desvantagens e condições de aplicabilidade distintas.

In [ ]:
# Importações usadas em todo o notebook
import numpy as np
import matplotlib.pyplot as plt

# Paleta de cores consistente para usarmos em todos os gráficos do notebook
COR_FUNCAO = "#1f3b73"      # azul escuro: a função f(x)
COR_RAIZ = "#d62728"        # vermelho: a raiz / ponto final
COR_ITER = "#2ca02c"        # verde: pontos de iteração
COR_SECANTE = "#ff7f0e"     # laranja: retas secantes / cordas
COR_TANGENTE = "#9467bd"    # roxo: retas tangentes
COR_AUX = "#7f7f7f"         # cinza: linhas auxiliares (eixos, guias)

plt.rcParams["figure.dpi"] = 100
plt.rcParams["font.size"] = 11

print("Bibliotecas importadas com sucesso.")

## 2. Fundamentação teórica: Teorema do Valor Médio e Teorema de Bolzano

Antes de construir qualquer método numérico, precisamos de uma garantia teórica de que **a raiz existe** em algum intervalo. É isso que o **Teorema de Bolzano** (também chamado de Teorema do Valor Intermediário, TVI) nos dá, e o **Teorema do Valor Médio** (TVM) nos ajuda a entender geometricamente.

### Teorema do Valor Médio (TVM)

> Se $f$ é contínua em $[a,b]$ e derivável em $(a,b)$, então existe pelo menos um ponto $c \in (a,b)$ tal que
> $$f'(c) = \frac{f(b)-f(a)}{b-a}.$$

**Intuição geométrica:** o lado direito da equação é a inclinação (coeficiente angular) da reta que liga os pontos $(a, f(a))$ e $(b, f(b))$ — a **corda** do gráfico de $f$. O TVM diz que existe pelo menos um ponto no interior do intervalo onde a **reta tangente** à curva é **paralela** a essa corda. Veja a figura abaixo.

### Teorema de Bolzano (caso particular relevante para zeros de funções)

> Se $f$ é contínua em $[a,b]$ e $f(a)$ e $f(b)$ têm **sinais opostos** (ou seja, $f(a)\cdot f(b) < 0$), então existe pelo menos uma raiz $x^* \in (a,b)$ tal que $f(x^*) = 0$.

Este é um caso particular do Teorema do Valor Intermediário: como $f$ é contínua e passa de um valor negativo para um positivo (ou vice-versa), ela necessariamente **cruza o zero** em algum ponto do intervalo — não é possível "pular" por cima do eixo $x$ sem tocá-lo, pois isso quebraria a continuidade.

**Por que isso importa para nós?** Todo método que veremos a seguir (dicotomia, falsa posição) parte de um intervalo $[a,b]$ onde $f(a)$ e $f(b)$ têm sinais opostos — essa é a condição de Bolzano, que **garante a existência de ao menos uma raiz** no intervalo antes mesmo de começarmos a iterar. Sem essa garantia, estaríamos "procurando no escuro".

In [ ]:
# Vamos ilustrar o TVM com uma função arbitrária, só para fins didáticos
# (não precisa ter relação com zeros de função aqui -- é só para ver a reta tangente paralela à corda)
def f_tvm(x):
    return 0.3 * x**3 - 1.5 * x**2 + x + 4

def df_tvm(x):
    # derivada analítica de f_tvm, calculada à mão
    return 0.9 * x**2 - 3.0 * x + 1

a, b = 0.2, 4.5

# Procuramos (numericamente, por busca em grade fina) o ponto c em (a,b) onde f'(c) = inclinação da corda
inclinacao_corda = (f_tvm(b) - f_tvm(a)) / (b - a)
x_fino = np.linspace(a, b, 20000)
c = x_fino[np.argmin(np.abs(df_tvm(x_fino) - inclinacao_corda))]

x_plot = np.linspace(a - 0.3, b + 0.3, 400)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x_plot, f_tvm(x_plot), color=COR_FUNCAO, lw=2, label=r"$f(x)$")

# a corda ligando (a, f(a)) e (b, f(b))
ax.plot([a, b], [f_tvm(a), f_tvm(b)], color=COR_SECANTE, lw=1.8, ls="--", label="Corda entre $(a,f(a))$ e $(b,f(b))$")
ax.scatter([a, b], [f_tvm(a), f_tvm(b)], color=COR_SECANTE, zorder=5)

# a reta tangente em c, com a mesma inclinação da corda
x_tan = np.linspace(c - 1.0, c + 1.0, 50)
y_tan = f_tvm(c) + inclinacao_corda * (x_tan - c)
ax.plot(x_tan, y_tan, color=COR_TANGENTE, lw=1.8, label="Tangente em $c$ (paralela à corda)")
ax.scatter([c], [f_tvm(c)], color=COR_TANGENTE, zorder=5, s=70)

ax.annotate("$a$", (a, f_tvm(a)), textcoords="offset points", xytext=(-15, -5))
ax.annotate("$b$", (b, f_tvm(b)), textcoords="offset points", xytext=(8, -5))
ax.annotate("$c$", (c, f_tvm(c)), textcoords="offset points", xytext=(8, 8))

ax.set_title("Teorema do Valor Médio: a tangente em $c$ é paralela à corda $[a,b]$")
ax.set_xlabel("$x$")
ax.set_ylabel("$f(x)$")
ax.legend(loc="best")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Inclinação da corda: {inclinacao_corda:.4f}")
print(f"Ponto c encontrado: {c:.4f}, com f'(c) = {df_tvm(c):.4f}")

## 3. Método da Dicotomia (Bisseção)

### Teoria

É o método mais simples e robusto para encontrar zeros de funções. Parte de um intervalo $[a,b]$ que satisfaz a condição de Bolzano, $f(a)\cdot f(b) < 0$, e o divide ao meio repetidamente, sempre mantendo a metade que contém a raiz.

**Algoritmo:**
1. Calcule o ponto médio $x_m = \dfrac{a+b}{2}$.
2. Calcule $f(x_m)$.
3. Se $f(a) \cdot f(x_m) < 0$, a raiz está em $[a, x_m]$ → faça $b \leftarrow x_m$.
   Caso contrário, a raiz está em $[x_m, b]$ → faça $a \leftarrow x_m$.
4. Repita até que o intervalo $(b-a)$ (ou $|f(x_m)|$) seja menor que a tolerância desejada.

**Pseudocódigo:**
```
enquanto (b - a) / 2 > tol:
    xm = (a + b) / 2
    se f(a) * f(xm) < 0:
        b = xm
    senão:
        a = xm
retorna (a + b) / 2
```

**Propriedades importantes:**
- **Sempre converge**, desde que $f$ seja contínua e $f(a)\cdot f(b) < 0$ — é o método mais robusto de todos.
- A cada iteração, o tamanho do intervalo **cai pela metade**: $|b_n - a_n| = \dfrac{|b_0-a_0|}{2^n}$. Isso significa convergência **linear** (um "dígito binário correto" por iteração), ou seja, relativamente **lenta** se comparada aos métodos que veremos a seguir.
- Não precisa calcular derivadas — só avaliações de $f$.

In [ ]:
# Função de teste que usaremos ao longo de toda a seção de dicotomia e falsa posição:
# f(x) = x^3 - x - 2, que tem uma única raiz real perto de x = 1.52
def f(x):
    return x**3 - x - 2

def df(x):
    # derivada analítica de f, usada mais à frente no método de Newton
    return 3 * x**2 - 1


def bissecao(f, a, b, tol=1e-6, max_iter=100):
    """
    Método da dicotomia (bisseção) para encontrar uma raiz de f em [a, b].

    Retorna a raiz aproximada e um histórico (lista de dicionários) com
    a, b, xm e f(xm) em cada iteração -- útil para visualizar o processo depois.
    """
    assert f(a) * f(b) < 0, "f(a) e f(b) devem ter sinais opostos (condição de Bolzano)."

    historico = []
    for i in range(1, max_iter + 1):
        xm = (a + b) / 2          # ponto médio do intervalo atual
        fxm = f(xm)
        historico.append({"iter": i, "a": a, "b": b, "xm": xm, "fxm": fxm})

        # critério de parada: intervalo já é menor que a tolerância
        if (b - a) / 2 < tol or fxm == 0:
            break

        # decide qual metade do intervalo mantém a mudança de sinal
        if f(a) * fxm < 0:
            b = xm
        else:
            a = xm

    return xm, historico


raiz_bissecao, hist_bissecao = bissecao(f, a=1.0, b=2.0, tol=1e-8)
print(f"Raiz encontrada pela bisseção: {raiz_bissecao:.8f}")
print(f"f(raiz) = {f(raiz_bissecao):.2e}")
print(f"Número de iterações: {len(hist_bissecao)}")

In [ ]:
# Visualização passo-a-passo: o intervalo [a,b] encolhendo a cada iteração até convergir
n_passos_mostrar = 6  # mostramos apenas as primeiras iterações para não poluir a figura
x_plot = np.linspace(0.8, 2.1, 400)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i >= len(hist_bissecao) or i >= n_passos_mostrar:
        ax.axis("off")
        continue
    passo = hist_bissecao[i]
    a_i, b_i, xm_i = passo["a"], passo["b"], passo["xm"]

    ax.plot(x_plot, f(x_plot), color=COR_FUNCAO, lw=1.8)
    ax.axhline(0, color="black", lw=0.8)

    # destaca o intervalo atual [a_i, b_i]
    ax.axvspan(a_i, b_i, color=COR_ITER, alpha=0.15)
    ax.scatter([a_i, b_i], [f(a_i), f(b_i)], color=COR_SECANTE, zorder=5, label="$a, b$")
    ax.scatter([xm_i], [f(xm_i)], color=COR_ITER, zorder=6, s=70, label="$x_m$")
    ax.axvline(xm_i, color=COR_ITER, ls="--", lw=1)

    ax.set_title(f"Iteração {passo['iter']}:  $x_m$ = {xm_i:.5f}")
    ax.set_xlim(0.8, 2.1)
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(loc="upper left", fontsize=8)

fig.suptitle("Método da Dicotomia: o intervalo $[a,b]$ encolhe a cada iteração", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico de convergência: erro (em relação à raiz "verdadeira", obtida com tolerância bem apertada) vs. iteração
raiz_referencia, _ = bissecao(f, a=1.0, b=2.0, tol=1e-14)  # raiz de altíssima precisão para servir de referência

erros_bissecao = [abs(passo["xm"] - raiz_referencia) for passo in hist_bissecao]

fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogy(range(1, len(erros_bissecao) + 1), erros_bissecao, "o-", color=COR_ITER, label="Erro $|x_m - x^*|$")
ax.set_xlabel("Iteração")
ax.set_ylabel("Erro absoluto (escala log)")
ax.set_title("Convergência do Método da Dicotomia")
ax.grid(alpha=0.3, which="both")
ax.legend()
plt.tight_layout()
plt.show()

print("Note que o erro decai de forma aproximadamente linear em escala log — cada iteração")
print("reduz o erro por um fator ~2 (convergência linear, não exponencialmente rápida).")

## 4. Método da Falsa Posição (Regula Falsi)

### Teoria

A ideia é parecida com a bisseção — mantemos um intervalo $[a,b]$ com $f(a)\cdot f(b)<0$ — mas, em vez de usar o **ponto médio** (que ignora os valores de $f$), usamos a **interseção da reta secante** que liga $(a, f(a))$ a $(b, f(b))$ com o eixo $x$. Essa escolha é mais "inteligente": se $f(a)$ está muito mais próximo de zero que $f(b)$, é razoável que a raiz esteja mais perto de $a$ do que do meio do intervalo.

A fórmula do ponto de falsa posição é:

$$x_f = a - f(a)\cdot\frac{b-a}{f(b)-f(a)} = \frac{a\,f(b) - b\,f(a)}{f(b) - f(a)}$$

**Algoritmo:** idêntico à bisseção, trocando apenas o cálculo do ponto candidato:
```
enquanto |f(xf)| > tol:
    xf = (a*f(b) - b*f(a)) / (f(b) - f(a))
    se f(a) * f(xf) < 0:
        b = xf
    senão:
        a = xf
retorna xf
```

**Diferença geométrica em relação à dicotomia:** a dicotomia sempre bissecta o intervalo ao meio (ponto médio geométrico), independente dos valores de $f$. A falsa posição usa a **reta secante** (corda) entre os extremos, de modo que o novo ponto é um "meio ponderado" pelos valores da função — geralmente mais próximo da raiz.

**Cuidado:** em funções com forte curvatura (concavidade constante na região), um dos extremos pode ficar "preso" (nunca ser atualizado), fazendo o método convergir de um lado só e de forma mais lenta do que se esperaria. Ainda assim, a falsa posição costuma precisar de **menos iterações que a dicotomia** no mesmo problema, como veremos a seguir.

In [ ]:
def falsa_posicao(f, a, b, tol=1e-6, max_iter=100):
    """
    Método da falsa posição (regula falsi) para encontrar uma raiz de f em [a, b].

    Retorna a raiz aproximada e um histórico com a, b, xf e f(xf) em cada iteração.
    """
    assert f(a) * f(b) < 0, "f(a) e f(b) devem ter sinais opostos (condição de Bolzano)."

    historico = []
    fxf = None
    for i in range(1, max_iter + 1):
        # ponto de interseção da secante entre (a, f(a)) e (b, f(b)) com o eixo x
        xf = (a * f(b) - b * f(a)) / (f(b) - f(a))
        fxf = f(xf)
        historico.append({"iter": i, "a": a, "b": b, "xf": xf, "fxf": fxf})

        # critério de parada por valor da função (mais natural aqui, pois o intervalo
        # pode não encolher simetricamente como na bisseção)
        if abs(fxf) < tol:
            break

        if f(a) * fxf < 0:
            b = xf
        else:
            a = xf

    return xf, historico


raiz_falsa_pos, hist_falsa_pos = falsa_posicao(f, a=1.0, b=2.0, tol=1e-10)
print(f"Raiz encontrada pela falsa posição: {raiz_falsa_pos:.8f}")
print(f"f(raiz) = {f(raiz_falsa_pos):.2e}")
print(f"Número de iterações: {len(hist_falsa_pos)}")

In [ ]:
# Visualização passo-a-passo: a secante e o novo ponto a cada iteração
n_passos_mostrar = 4
x_plot = np.linspace(0.8, 2.1, 400)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for i, ax in enumerate(axes):
    if i >= len(hist_falsa_pos) or i >= n_passos_mostrar:
        ax.axis("off")
        continue
    passo = hist_falsa_pos[i]
    a_i, b_i, xf_i = passo["a"], passo["b"], passo["xf"]

    ax.plot(x_plot, f(x_plot), color=COR_FUNCAO, lw=1.8)
    ax.axhline(0, color="black", lw=0.8)

    # a reta secante entre (a_i, f(a_i)) e (b_i, f(b_i))
    ax.plot([a_i, b_i], [f(a_i), f(b_i)], color=COR_SECANTE, lw=1.6, ls="--", label="Secante")
    ax.scatter([a_i, b_i], [f(a_i), f(b_i)], color=COR_SECANTE, zorder=5)
    ax.scatter([xf_i], [f(xf_i)], color=COR_ITER, zorder=6, s=70, label="$x_f$ (nova aprox.)")
    ax.axvline(xf_i, color=COR_ITER, ls=":", lw=1)

    ax.set_title(f"Iteração {passo['iter']}\n$x_f$ = {xf_i:.5f}")
    ax.set_xlim(0.8, 2.1)
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(loc="upper left", fontsize=8)

fig.suptitle("Método da Falsa Posição: a secante aponta diretamente para perto da raiz", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Comparação de velocidade de convergência: dicotomia vs. falsa posição, no mesmo problema
erros_falsa_pos = [abs(passo["xf"] - raiz_referencia) for passo in hist_falsa_pos]

fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogy(range(1, len(erros_bissecao) + 1), erros_bissecao, "o-", color=COR_ITER, label="Dicotomia")
ax.semilogy(range(1, len(erros_falsa_pos) + 1), erros_falsa_pos, "s-", color=COR_SECANTE, label="Falsa Posição")
ax.set_xlabel("Iteração")
ax.set_ylabel("Erro absoluto (escala log)")
ax.set_title("Dicotomia vs. Falsa Posição: mesmo problema, $f(x) = x^3 - x - 2$")
ax.grid(alpha=0.3, which="both")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Dicotomia precisou de {len(hist_bissecao)} iterações.")
print(f"Falsa posição precisou de {len(hist_falsa_pos)} iterações.")
print("Neste exemplo, a falsa posição converge bem mais rápido, pois a curvatura de f")
print("favorece a aproximação pela secante.")

## 5. Método de Newton-Raphson

### Teoria

O método de Newton usa informação local de **derivada** para construir aproximações muito mais eficientes. A ideia pode ser deduzida da expansão em série de Taylor de $f$ em torno de um ponto $x_n$:

$$f(x) \approx f(x_n) + f'(x_n)(x - x_n)$$

Se aproximarmos $f$ pela sua reta tangente em $x_n$ e procurarmos onde **essa reta** cruza o eixo $x$ (ou seja, resolvendo $0 = f(x_n) + f'(x_n)(x_{n+1}-x_n)$), obtemos a fórmula de iteração:

$$x_{n+1} = x_n - \frac{f(x_n)}{f'(x_n)}$$

**Interpretação geométrica:** a cada passo, traçamos a reta tangente ao gráfico de $f$ no ponto atual $(x_n, f(x_n))$ e tomamos como próxima aproximação o ponto onde essa tangente cruza o eixo $x$. Repetimos o processo até que $x_n$ esteja suficientemente próximo da raiz.

**Critério de parada por precisão pré-fixada:** na prática, paramos quando:
- $|f(x_n)| < \varepsilon_1$ (o valor da função já é próximo de zero), **ou**
- $|x_{n+1} - x_n| < \varepsilon_2$ (a variável já não está mudando significativamente),

onde $\varepsilon_1,\varepsilon_2$ são tolerâncias pré-fixadas escolhidas conforme a precisão desejada na aplicação.

**Convergência quadrática:** quando $x_0$ está suficientemente próximo da raiz $x^*$ e $f'(x^*) \neq 0$, pode-se mostrar que o erro $e_n = x_n - x^*$ satisfaz, aproximadamente,

$$e_{n+1} \approx \frac{f''(x^*)}{2f'(x^*)}\, e_n^2 ,$$

ou seja, o erro da iteração seguinte é proporcional ao **quadrado** do erro atual. Isso é chamado de **convergência quadrática**: o número de dígitos corretos aproximadamente **dobra a cada iteração** — muito mais rápido que a convergência linear da dicotomia.

**Limitações:** o método de Newton exige o cálculo de $f'(x)$ (nem sempre disponível/fácil) e **não é globalmente garantido**: pode divergir, oscilar ou "explodir" se:
- o ponto inicial $x_0$ estiver longe da raiz (fora da região de convergência);
- $f'(x_n) \approx 0$ em algum ponto da iteração (a tangente fica quase horizontal e "atira" o próximo ponto para longe);
- a função tiver pontos de inflexão ou múltiplas raízes próximas que confundem a tangente.

In [ ]:
def newton(f, df, x0, tol_f=1e-10, tol_x=1e-12, max_iter=50):
    """
    Método de Newton-Raphson com critério de parada por precisão pré-fixada:
    para quando |f(x)| < tol_f OU |x_{n+1} - x_n| < tol_x.

    Retorna a raiz aproximada e um histórico com x, f(x) e f'(x) em cada iteração.
    """
    x = x0
    historico = []
    for i in range(1, max_iter + 1):
        fx = f(x)
        dfx = df(x)

        if dfx == 0:
            raise ZeroDivisionError(f"Derivada nula em x = {x:.6f}; o método de Newton não pode prosseguir.")

        x_novo = x - fx / dfx   # interseção da reta tangente com o eixo x
        historico.append({"iter": i, "x": x, "fx": fx, "dfx": dfx, "x_novo": x_novo})

        # critério de parada por precisão pré-fixada (valor da função OU variação de x)
        if abs(fx) < tol_f or abs(x_novo - x) < tol_x:
            x = x_novo
            break

        x = x_novo

    return x, historico


raiz_newton, hist_newton = newton(f, df, x0=1.5, tol_f=1e-12)
print(f"Raiz encontrada por Newton: {raiz_newton:.12f}")
print(f"f(raiz) = {f(raiz_newton):.2e}")
print(f"Número de iterações: {len(hist_newton)}")

In [ ]:
# Visualização: as tangentes sucessivas convergindo à raiz
x_plot = np.linspace(0.8, 2.1, 400)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(x_plot, f(x_plot), color=COR_FUNCAO, lw=2, label="$f(x)$", zorder=3)
ax.axhline(0, color="black", lw=0.8)

cores_iter = plt.cm.viridis(np.linspace(0.1, 0.85, len(hist_newton)))

for i, (passo, cor) in enumerate(zip(hist_newton, cores_iter)):
    x_i, fx_i, dfx_i, x_novo_i = passo["x"], passo["fx"], passo["dfx"], passo["x_novo"]

    # reta tangente em x_i: y = f(x_i) + f'(x_i) * (x - x_i)
    x_tan = np.linspace(x_i - 0.35, x_i + 0.35, 20)
    y_tan = fx_i + dfx_i * (x_tan - x_i)
    ax.plot(x_tan, y_tan, color=cor, lw=1.3, ls="--")

    ax.scatter([x_i], [fx_i], color=cor, zorder=5, s=55)
    ax.annotate(f"$x_{i}$", (x_i, fx_i), textcoords="offset points", xytext=(5, 8), fontsize=9, color=cor)
    ax.plot([x_i, x_i], [0, fx_i], color=cor, lw=0.7, ls=":")

ax.scatter([raiz_newton], [f(raiz_newton)], color=COR_RAIZ, zorder=6, s=90, marker="*", label=f"Raiz $\\approx$ {raiz_newton:.6f}")

ax.set_xlim(0.8, 2.1)
ax.set_ylim(-4, 6)
ax.set_title("Método de Newton: tangentes sucessivas convergindo à raiz")
ax.set_xlabel("$x$")
ax.set_ylabel("$f(x)$")
ax.grid(alpha=0.3)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Demonstração da convergência quadrática: o número de dígitos corretos dobra a cada iteração
erros_newton = [abs(passo["x"] - raiz_referencia) for passo in hist_newton]
# adicionamos também o erro do último x (pós-convergência) para fechar a sequência
erros_newton.append(abs(hist_newton[-1]["x_novo"] - raiz_referencia))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# painel 1: erro em escala semilog (eixo y log, eixo x linear)
axes[0].semilogy(range(len(erros_newton)), erros_newton, "o-", color=COR_TANGENTE)
axes[0].set_xlabel("Iteração")
axes[0].set_ylabel("Erro absoluto (escala log)")
axes[0].set_title("Convergência de Newton (semilog)")
axes[0].grid(alpha=0.3, which="both")

# painel 2: log(erro_{n+1}) vs log(erro_n) -- se a inclinação for ~2, confirma convergência quadrática
erros_validos = [e for e in erros_newton if e > 1e-15]  # evita log(0)
if len(erros_validos) >= 2:
    log_en = np.log10(erros_validos[:-1])
    log_en1 = np.log10(erros_validos[1:])
    axes[1].plot(log_en, log_en1, "o-", color=COR_RAIZ, label="Dados observados")
    # reta de referência com inclinação 2 (convergência quadrática ideal), passando pelo primeiro ponto
    if len(log_en) > 0:
        x_ref = np.array([log_en.min(), log_en.max()])
        y_ref = 2 * (x_ref - log_en[0]) + log_en1[0]
        axes[1].plot(x_ref, y_ref, "--", color=COR_AUX, label="Inclinação 2 (quadrática ideal)")
axes[1].set_xlabel(r"$\log_{10}(e_n)$")
axes[1].set_ylabel(r"$\log_{10}(e_{n+1})$")
axes[1].set_title("Inclinação ≈ 2 confirma convergência quadrática")
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print("Dígitos corretos (aproximado) por iteração:")
for i, e in enumerate(erros_newton):
    digitos = -np.log10(e) if e > 0 else float("inf")
    print(f"  iteração {i}: erro = {e:.3e}  (~{digitos:.1f} dígitos corretos)")

### Quando Newton falha

Apesar de sua velocidade, o método de Newton não é "mágico": a convergência depende criticamente da escolha do ponto inicial $x_0$ e do comportamento de $f'$ ao longo da iteração. Vamos ilustrar dois tipos clássicos de falha usando $g(x) = x^3 - 2x + 2$ (que tem uma raiz real perto de $x \approx -1.77$):

1. **Ponto inicial ruim causando ciclo/divergência:** partindo de $x_0 = 0$, a tangente em $x_0$ tem inclinação $g'(0) = -2$ e "joga" a iteração para longe da raiz; o processo entra em um ciclo e nunca converge.
2. **Derivada nula/quase nula:** se, em algum ponto da iteração, $f'(x_n)$ estiver muito próximo de zero, a tangente fica quase horizontal e o próximo ponto $x_{n+1} = x_n - f(x_n)/f'(x_n)$ é "lançado" para muito longe, podendo levar a divergência numérica (overflow) ou a uma região completamente diferente do domínio.

In [ ]:
# Caso de falha 1: ponto inicial ruim leva a um ciclo (não converge)
def g(x):
    return x**3 - 2 * x + 2

def dg(x):
    return 3 * x**2 - 2

_, hist_newton_falha = newton(g, dg, x0=0.0, tol_f=1e-12, max_iter=10)

print("Iterações de Newton para g(x) = x^3 - 2x + 2, partindo de x0 = 0:")
for passo in hist_newton_falha:
    print(f"  iter {passo['iter']}: x = {passo['x']:.6f} -> x_novo = {passo['x_novo']:.6f}  (f(x) = {passo['fx']:.6f})")
print("\nO método fica preso num ciclo entre x=0 e x=1, nunca convergindo para a raiz real")
print("(que fica perto de x ≈ -1.7693). Isso mostra que a escolha de x0 é crucial.")

print("\n" + "=" * 70)

# Caso de falha 2: derivada (quase) nula faz a iteração "explodir"
# g'(x) = 3x^2 - 2 = 0 em x = sqrt(2/3) ≈ 0.8165 -- um ponto inicial próximo disso é perigoso
x0_perigoso = np.sqrt(2 / 3) * 1.0001  # bem próximo do ponto onde g'(x) = 0
try:
    _, hist_perigoso = newton(g, dg, x0=x0_perigoso, tol_f=1e-10, max_iter=6)
    print(f"Partindo de x0 = {x0_perigoso:.6f} (próximo de g'(x)=0):")
    for passo in hist_perigoso:
        print(f"  iter {passo['iter']}: x = {passo['x']:.4f} -> x_novo = {passo['x_novo']:.4f}  (f'(x) = {passo['dfx']:.6f})")
    print("\nRepare como o passo seguinte é lançado para muito longe da região original,")
    print("pois estamos dividindo por um f'(x) muito próximo de zero.")
except ZeroDivisionError as erro:
    print(f"Erro capturado: {erro}")

## 6. Método das Aproximações Sucessivas (Ponto Fixo)

### Teoria

A ideia aqui é reescrever a equação $f(x) = 0$ em uma forma equivalente

$$x = g(x)$$

para alguma função $g$ (existem infinitas formas de fazer isso — basta somar $x$ aos dois lados de $f(x)=0$ de maneiras diferentes, por exemplo). Uma solução $x^*$ dessa equação é chamada de **ponto fixo** de $g$, pois $g(x^*) = x^*$ — e note que todo ponto fixo de $g$ é também uma raiz de $f$.

A partir de um palpite inicial $x_0$, construímos a sequência iterativa:

$$x_{n+1} = g(x_n)$$

e esperamos que ela convirja para $x^*$.

**Critério de convergência:** pelo Teorema do Ponto Fixo, se $g$ é continuamente derivável numa vizinhança de $x^*$ e

$$|g'(x)| < 1 \quad \text{para todo } x \text{ nessa vizinhança},$$

então a iteração converge para $x^*$ para qualquer $x_0$ suficientemente próximo. Quanto menor $|g'(x^*)|$, mais rápida a convergência. Se $|g'(x^*)| > 1$, a iteração **diverge** (afasta-se da raiz) mesmo partindo de um ponto próximo.

**Visualização "teia de aranha" (cobweb plot):** uma forma clássica de visualizar essa iteração é desenhar, no mesmo gráfico, a curva $y=g(x)$ e a reta $y=x$. Partindo de $x_0$:
1. Suba (ou desça) verticalmente até encontrar a curva $y=g(x)$: isso dá $g(x_0)$.
2. Mova horizontalmente até a reta $y=x$: isso "transporta" o valor $g(x_0)$ para o eixo $x$, tornando-o o novo $x_1$.
3. Repita o processo.

O resultado visual lembra uma teia de aranha (ou uma escada) convergindo para o ponto onde $y=g(x)$ cruza $y=x$ — exatamente o ponto fixo $x^*$.

In [ ]:
def ponto_fixo(g, x0, tol=1e-8, max_iter=50):
    """
    Método das aproximações sucessivas (ponto fixo): x_{n+1} = g(x_n).

    Retorna a sequência completa de iterados (incluindo x0), parando quando
    |x_{n+1} - x_n| < tol ou ao atingir max_iter (útil para visualizar também
    o caso divergente, que nunca satisfaz o critério de parada).
    """
    xs = [x0]
    for _ in range(max_iter):
        x_novo = g(xs[-1])
        xs.append(x_novo)
        if abs(xs[-1] - xs[-2]) < tol or not np.isfinite(x_novo) or abs(x_novo) > 1e6:
            break
    return xs


# f(x) = x^3 - x - 2 = 0  pode ser reescrita de (pelo menos) duas formas diferentes:

# Forma 1 (CONVERGENTE): a partir de x^3 = x + 2, isolamos x = (x+2)^(1/3)
def g_convergente(x):
    return np.cbrt(x + 2)

def dg_convergente(x):
    return (1 / 3) * (x + 2) ** (-2 / 3)

# Forma 2 (DIVERGENTE): a partir de x^3 - 2 = x, isolamos x = x^3 - 2
def g_divergente(x):
    return x**3 - 2

def dg_divergente(x):
    return 3 * x**2


xs_convergente = ponto_fixo(g_convergente, x0=1.0, tol=1e-10, max_iter=30)
xs_divergente = ponto_fixo(g_divergente, x0=1.3, tol=1e-10, max_iter=8)

print(f"g'(x*) na forma convergente: {dg_convergente(raiz_referencia):.4f}  (|g'| < 1 -> converge)")
print(f"Ponto fixo (convergente) após {len(xs_convergente)-1} iterações: {xs_convergente[-1]:.8f}")
print()
print(f"g'(x*) na forma divergente:  {dg_divergente(raiz_referencia):.4f}  (|g'| > 1 -> diverge)")
print(f"Sequência divergente (primeiros termos): {[round(v, 4) for v in xs_divergente[:6]]}")

In [ ]:
def plot_cobweb(g, xs, x_range, ax, titulo):
    """
    Desenha o gráfico "teia de aranha" (cobweb plot) da iteração x_{n+1} = g(x_n):
    a curva y = g(x), a reta y = x, e os segmentos em degrau que representam a iteração.
    """
    x_plot = np.linspace(*x_range, 400)
    ax.plot(x_plot, g(x_plot), color=COR_FUNCAO, lw=2, label="$y = g(x)$")
    ax.plot(x_plot, x_plot, color=COR_AUX, lw=1.2, ls="--", label="$y = x$")

    # desenha os degraus: (x_n, x_n) -> (x_n, x_{n+1}) -> (x_{n+1}, x_{n+1})
    for n in range(len(xs) - 1):
        x_n, x_n1 = xs[n], xs[n + 1]
        ax.plot([x_n, x_n], [x_n, x_n1], color=COR_ITER, lw=1.0)
        ax.plot([x_n, x_n1], [x_n1, x_n1], color=COR_ITER, lw=1.0)

    ax.scatter(xs[0], xs[0], color=COR_SECANTE, zorder=5, s=50, label="$x_0$")
    ax.set_title(titulo)
    ax.set_xlabel("$x_n$")
    ax.set_ylabel("$x_{n+1}$")
    # fixamos os limites explicitamente: no caso divergente a sequência explode rapidamente
    # e, sem isso, o autoscale do matplotlib esticaria o gráfico até ficar ilegível
    ax.set_xlim(*x_range)
    ax.set_ylim(*x_range)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)


fig, axes = plt.subplots(1, 2, figsize=(13, 6))

plot_cobweb(g_convergente, xs_convergente, (0.5, 2.2), axes[0],
            f"Caso CONVERGENTE: $g(x)=\\sqrt[3]{{x+2}}$,  $|g'(x^*)|\\approx{abs(dg_convergente(raiz_referencia)):.3f} < 1$")

plot_cobweb(g_divergente, xs_divergente, (-3, 3), axes[1],
            f"Caso DIVERGENTE: $g(x)=x^3-2$,  $|g'(x^*)|\\approx{abs(dg_divergente(raiz_referencia)):.2f} > 1$")

plt.tight_layout()
plt.show()

print("No painel esquerdo, a 'escada' espirala suavemente até o ponto fixo (interseção das curvas).")
print("No painel direito, cada passo se afasta mais da raiz -- a teia 'explode' para fora,")
print("ilustrando concretamente por que a condição |g'(x*)| < 1 é necessária para convergência.")

## 7. Comparação final entre os métodos

Agora vamos colocar todos os quatro métodos lado a lado, resolvendo o **mesmo problema**, para visualizar de forma direta a diferença de velocidade de convergência entre eles.

Vamos usar $f(x) = \cos(x) - x$, cuja raiz está em $x^* \approx 0.739085$ (o "número de Dottie", um clássico exemplo de zero de função transcendental — não há como isolar $x$ algebricamente nessa equação).

Para o ponto fixo, usamos a forma natural $x = \cos(x)$, ou seja $g(x) = \cos(x)$, que satisfaz $|g'(x^*)| = |\sin(x^*)| \approx 0.674 < 1$ — logo converge.

In [ ]:
# Problema de teste comum a todos os métodos: f(x) = cos(x) - x
def f2(x):
    return np.cos(x) - x

def df2(x):
    return -np.sin(x) - 1

def g2(x):
    return np.cos(x)

# raiz de referência de altíssima precisão (via bisseção com tolerância bem apertada)
raiz_ref2, _ = bissecao(f2, a=0.0, b=1.0, tol=1e-15)

# rodamos os quatro métodos no mesmo problema
_, hist_bis2 = bissecao(f2, a=0.0, b=1.0, tol=1e-12)
_, hist_fp2 = falsa_posicao(f2, a=0.0, b=1.0, tol=1e-14)
_, hist_newton2 = newton(f2, df2, x0=0.0, tol_f=1e-14, tol_x=1e-15)
xs_ptofixo2 = ponto_fixo(g2, x0=0.0, tol=1e-14, max_iter=100)

# extraímos a evolução do erro |aproximação - raiz_ref2| para cada método
erros_bis2 = [abs(p["xm"] - raiz_ref2) for p in hist_bis2]
erros_fp2 = [abs(p["xf"] - raiz_ref2) for p in hist_fp2]
erros_newton2 = [abs(p["x"] - raiz_ref2) for p in hist_newton2]
erros_ptofixo2 = [abs(x - raiz_ref2) for x in xs_ptofixo2]

fig, ax = plt.subplots(figsize=(8, 6))
ax.semilogy(range(1, len(erros_bis2) + 1), erros_bis2, "o-", color=COR_ITER, label=f"Dicotomia ({len(erros_bis2)} iter.)")
ax.semilogy(range(1, len(erros_fp2) + 1), erros_fp2, "s-", color=COR_SECANTE, label=f"Falsa Posição ({len(erros_fp2)} iter.)")
ax.semilogy(range(1, len(erros_newton2) + 1), erros_newton2, "^-", color=COR_TANGENTE, label=f"Newton ({len(erros_newton2)} iter.)")
ax.semilogy(range(len(erros_ptofixo2)), erros_ptofixo2, "d-", color=COR_RAIZ, label=f"Ponto Fixo ({len(erros_ptofixo2)-1} iter.)")

ax.set_xlabel("Iteração")
ax.set_ylabel("Erro absoluto $|x_n - x^*|$ (escala log)")
ax.set_title(r"Comparação de convergência: $f(x) = \cos(x) - x$")
ax.grid(alpha=0.3, which="both")
ax.legend()
plt.tight_layout()
plt.show()

print("Número de iterações até a tolerância escolhida em cada método:")
print(f"  Dicotomia:      {len(erros_bis2)}")
print(f"  Falsa Posição:  {len(erros_fp2)}")
print(f"  Newton:         {len(erros_newton2)}")
print(f"  Ponto Fixo:     {len(erros_ptofixo2)-1}")
print("\nFica evidente a hierarquia de velocidade: Newton converge muitíssimo mais rápido")
print("(convergência quadrática), seguido por falsa posição e ponto fixo, com a dicotomia")
print("sendo a mais lenta (convergência linear, mas a mais robusta das quatro).")

## 8. Para ir além: resumo comparativo

| Método | Precisa de $f'(x)$? | Garantia de convergência | Velocidade (ordem) | Robustez | Observações |
|---|---|---|---|---|---|
| **Dicotomia (Bisseção)** | Não | Sempre converge, se $f(a)f(b)<0$ | Linear (ordem 1) — lenta, mas previsível | Muito alta | O "porto seguro": use quando não há garantias sobre $f$ além da continuidade |
| **Falsa Posição** | Não | Sempre converge, se $f(a)f(b)<0$ | Superlinear em geral, mas pode "prender" um extremo e desacelerar | Alta | Normalmente mais rápida que a dicotomia; cuidado com funções muito assimétricas |
| **Newton-Raphson** | Sim | Local — depende de $x_0$ próximo da raiz e $f'(x^*)\neq 0$ | Quadrática (ordem 2) — dígitos corretos dobram a cada iteração | Baixa/média | O mais rápido quando converge; pode divergir, ciclar ou falhar com $f'(x_n)\approx 0$ |
| **Ponto Fixo** | Não (mas precisa de $g'$ para análise) | Local — depende de $\vert g'(x)\vert < 1$ perto da raiz | Linear, com taxa $\approx \vert g'(x^*)\vert$ | Baixa/média | A velocidade e até a convergência dependem fortemente de **como** você escolhe reescrever $f(x)=0$ como $x=g(x)$ |

**Resumo em uma frase para cada método:**
- *Dicotomia*: lenta, mas nunca falha — ótima escolha quando confiabilidade importa mais que velocidade.
- *Falsa posição*: um upgrade "quase de graça" sobre a dicotomia, usando a informação geométrica da secante.
- *Newton*: extremamente rápido perto da raiz, mas exige derivada e um bom chute inicial — use quando tiver essas duas coisas.
- *Ponto fixo*: conceitualmente simples e flexível, mas a escolha de $g(x)$ é crucial — uma escolha ruim pode fazer o método divergir mesmo que a raiz exista.

Na prática de engenharia, uma estratégia comum é **combinar métodos**: usar a dicotomia (ou falsa posição) para obter rapidamente uma boa aproximação inicial robusta, e então "polir" essa aproximação com algumas iterações de Newton para ganhar precisão rapidamente — aproveitando o melhor dos dois mundos (robustez + velocidade).